# Utilities

In [1]:
# detectors library
!pip install -q detectors datasets
!pip install -q compressai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.8/616.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))

In [3]:
import os
import random
import datetime
from dataclasses import dataclass
from typing import List, Optional, Callable, Dict, Any

import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

import detectors
import timm

import compressai
from compressai.zoo import cheng2020_attn

from io import BytesIO
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd

SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Using device: cuda


# Datasets

## (Loaders and normalisations)

In [4]:
# Choose dataset: "cifar10", "cifar100", or "imagenet"
DATASET_NAME = "cifar100"

if DATASET_NAME == "cifar10":
    DATA_MEAN = (0.4914, 0.4822, 0.4465)
    DATA_STD  = (0.2023, 0.1994, 0.2010)
    NUM_CLASSES = 10
elif DATASET_NAME == "cifar100":
    DATA_MEAN = (0.5071, 0.4867, 0.4408)
    DATA_STD  = (0.2675, 0.2565, 0.2761)
    NUM_CLASSES = 100
elif DATASET_NAME == "imagenet":
    DATA_MEAN = (0.485, 0.456, 0.406)
    DATA_STD  = (0.229, 0.224, 0.225)
    NUM_CLASSES = 1000
else:
    raise ValueError(f"Unsupported DATASET_NAME: {DATASET_NAME}")

DATA_MEAN_T = torch.tensor(DATA_MEAN, device=device).view(1, 3, 1, 1)
DATA_STD_T  = torch.tensor(DATA_STD, device=device).view(1, 3, 1, 1)

def denormalize(x: torch.Tensor) -> torch.Tensor:
    # normalised → [0,1]
    x = x * DATA_STD_T + DATA_MEAN_T
    return torch.clamp(x, 0.0, 1.0)

def renormalize(x: torch.Tensor) -> torch.Tensor:
    # [0,1] → normalised
    return (x - DATA_MEAN_T) / DATA_STD_T

def project_linf_pixel(x_adv_px: torch.Tensor, x0_px: torch.Tensor, eps: float) -> torch.Tensor:
    # Project to L-inf ball around x0 in pixel space
    x_adv_px = torch.max(torch.min(x_adv_px, x0_px + eps), x0_px - eps)
    # Clamp to valid pixel range
    return x_adv_px.clamp(0.0, 1.0)

In [5]:
import math

def batch_metrics_from_normalized(orig_norm: torch.Tensor,
                                  pert_norm: torch.Tensor):
    """
    MSE, MAE, and PSNR between two batches of normalised images.
    """
    # Go back to pixel space [0,1]
    x0 = denormalize(orig_norm)
    x1 = denormalize(pert_norm)

    # 1. MSE (Mean Squared Error) per image
    mse = torch.mean((x0 - x1) ** 2, dim=(1, 2, 3))

    # 2. MAE (Mean Absolute Error) per image
    mae = torch.mean(torch.abs(x0 - x1), dim=(1, 2, 3))

    # 3. PSNR
    eps = 1e-10
    mse_clamped = torch.clamp(mse, min=eps)
    psnr = 10.0 * torch.log10(1.0 / mse_clamped)  # max pixel value is 1.0

    return mse, mae, psnr

## Transformations

In [6]:
transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize(DATA_MEAN, DATA_STD),
])

@dataclass
class DataConfig:
    batch_size: int = 128
    num_workers: int = 2
    root: str = "./data"

class CifarDataModule:
    def __init__(self, config: DataConfig, dataset_name: str):
        self.config = config
        self.dataset_name = dataset_name

    def _get_dataset(self, train: bool, transform):
        if self.dataset_name == "cifar10":
            return torchvision.datasets.CIFAR10(
                root=self.config.root,
                train=train,
                download=True,
                transform=transform,
            )
        elif self.dataset_name == "cifar100":
            return torchvision.datasets.CIFAR100(
                root=self.config.root,
                train=train,
                download=True,
                transform=transform,
            )
        else:
            raise ValueError(f"Unsupported dataset: {self.dataset_name}")

    def _make_loader(self, train: bool, transform) -> DataLoader:
        dataset = self._get_dataset(train=train, transform=transform)
        return DataLoader(
            dataset,
            batch_size=self.config.batch_size,
            shuffle=train,
            num_workers=self.config.num_workers,
            pin_memory=True,
        )

    def test_loader(self):
        return self._make_loader(train=False, transform=transform_test)


class HFImageNetDataset(torch.utils.data.IterableDataset):
    """Wraps a HuggingFace streaming dataset as a PyTorch IterableDataset."""
    def __init__(self, hf_dataset, transform=None):
        self.hf_dataset = hf_dataset
        self.transform = transform

    def __iter__(self):
        for item in self.hf_dataset:
            image = item["image"].convert("RGB")
            label = item["label"]
            if self.transform:
                image = self.transform(image)
            yield image, label


class ImageNetDataModule:
    def __init__(self, config: DataConfig):
        self.config = config

    def test_loader(self):
        from datasets import load_dataset

        transform = T.Compose([
            T.Resize(256),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize(DATA_MEAN, DATA_STD),
        ])

        hf_stream = load_dataset(
            "imagenet-1k", split="validation", streaming=True
        )
        dataset = HFImageNetDataset(hf_stream, transform=transform)

        return DataLoader(
            dataset,
            batch_size=self.config.batch_size,
        )


if DATASET_NAME in ("cifar10", "cifar100"):
    data_cfg = DataConfig(batch_size=128, num_workers=2)
    data_module = CifarDataModule(data_cfg, DATASET_NAME)
elif DATASET_NAME == "imagenet":
    data_cfg = DataConfig(batch_size=32, num_workers=2)
    data_module = ImageNetDataModule(data_cfg)

base_test_loader = data_module.test_loader()

100%|██████████| 169M/169M [00:06<00:00, 26.4MB/s]


# Models

## Model Helper Functions

In [7]:
# --- Helper for resizing 32x32 CIFAR images to 224x224 for ViT ---
class ResizeWrapper(nn.Module):
    def __init__(self, model, target_size=224):
        super().__init__()
        self.model = model
        self.register_buffer("target_size", torch.tensor(target_size))

    def forward(self, x):
        # Resize input (B, C, 32, 32) -> (B, C, 224, 224)
        x_resized = F.interpolate(
            x, 
            size=(int(self.target_size), int(self.target_size)), 
            mode='bicubic', 
            align_corners=False
        )
        return self.model(x_resized)

## Registry

In [8]:
@dataclass
class ModelSpec:
    name: str
    loader: Callable[[], nn.Module]
    num_classes: int
    input_size: int = 32
    dataset: str = DATASET_NAME

def load_resnet18_cifar10() -> nn.Module:
    return timm.create_model("resnet18_cifar10", pretrained=True)

def load_resnet34_cifar10() -> nn.Module:
    return timm.create_model("resnet34_cifar10", pretrained=True)

def load_resnet50_cifar10() -> nn.Module:
    return timm.create_model("resnet50_cifar10", pretrained=True)

def load_resnet18_cifar100() -> nn.Module:
    return timm.create_model("resnet18_cifar100", pretrained=True)

def load_resnet34_cifar100() -> nn.Module:
    return timm.create_model("resnet34_cifar100", pretrained=True)

def load_resnet50_cifar100() -> nn.Module:
    return timm.create_model("resnet50_cifar100", pretrained=True)

def load_vit_base16_cifar10() -> nn.Module:
    model = timm.create_model("timm/vit_base_patch16_224.orig_in21k_ft_in1k", pretrained=False)
    model.head = nn.Linear(model.head.in_features, 10)
    model.load_state_dict(
        torch.hub.load_state_dict_from_url("https://huggingface.co/edadaltocg/vit_base_patch16_224_in21k_ft_cifar10/resolve/main/pytorch_model.bin",
            map_location="cpu",
            file_name="vit_base_patch16_224_in21k_ft_cifar10.pth",
        )
    )
    return ResizeWrapper(model, target_size=224)

def load_vit_base16_cifar100() -> nn.Module:
    model = timm.create_model("timm/vit_base_patch16_224.orig_in21k_ft_in1k", pretrained=False)
    model.head = nn.Linear(model.head.in_features, 100)
    model.load_state_dict(
        torch.hub.load_state_dict_from_url("https://huggingface.co/edadaltocg/vit_base_patch16_224_in21k_ft_cifar100/resolve/main/pytorch_model.bin",
            map_location="cpu",
            file_name="vit_base_patch16_224_in21k_ft_cifar100.pth",
        )
    )
    return ResizeWrapper(model, target_size=224)

def load_resnet50_imagenet() -> nn.Module:
    return torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)

MODEL_REGISTRY: Dict[str, ModelSpec] = {
    # CIFAR-10 models
    "resnet18_cifar10": ModelSpec(
        name="resnet18_cifar10",
        loader=load_resnet18_cifar10,
        num_classes=10,
        input_size=32,
        dataset="cifar10"
    ),
    "resnet34_cifar10": ModelSpec(
        name="resnet34_cifar10",
        loader=load_resnet34_cifar10,
        num_classes=10,
        input_size=32,
        dataset="cifar10"
    ),
    "resnet50_cifar10": ModelSpec(
        name="resnet50_cifar10",
        loader=load_resnet50_cifar10,
        num_classes=10,
        input_size=32,
        dataset="cifar10"
    ),

    # CIFAR-100 models
    "resnet18_cifar100": ModelSpec(
        name="resnet18_cifar100",
        loader=load_resnet18_cifar100,
        num_classes=100,
        input_size=32,
        dataset="cifar100"
    ),
    "resnet34_cifar100": ModelSpec(
        name="resnet34_cifar100",
        loader=load_resnet34_cifar100,
        num_classes=100,
        input_size=32,
        dataset="cifar100"
    ),
    "resnet50_cifar100": ModelSpec(
        name="resnet50_cifar100",
        loader=load_resnet50_cifar100,
        num_classes=100,
        input_size=32,
        dataset="cifar100"
    ),
    
    "vit_base16_cifar10": ModelSpec(
        name="vit_base16_cifar10",
        loader=load_vit_base16_cifar10,
        num_classes=10,
        input_size=32,
        dataset="cifar10"
    ),

    "vit_base16_cifar100": ModelSpec(
        name="vit_base16_cifar100",
        loader=load_vit_base16_cifar100,
        num_classes=100,
        input_size=32,
        dataset="cifar100"
    ),

    # ImageNet models
    "resnet50_imagenet": ModelSpec(
        name="resnet50_imagenet",
        loader=load_resnet50_imagenet,
        num_classes=1000,
        input_size=224,
        dataset="imagenet",
    ),
    
}

def build_model(spec_name: str, device: torch.device) -> nn.Module:
    spec = MODEL_REGISTRY[spec_name]
    if spec.dataset != DATASET_NAME:
        raise ValueError(
            f"Model {spec_name} is for {spec.dataset}, "
            f"but DATASET_NAME is {DATASET_NAME}"
        )

    model = spec.loader().to(device)
    model.eval()
    return model

# Perturbations

In [9]:
from abc import ABC, abstractmethod

class Perturbation(ABC):
    """
    Generic interface for any perturbation:
    - attacks (FGSM, PGD, ...)
    - compressions (JPEG, PCA, ...)
    """
    
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def apply(self, model: nn.Module, images: torch.Tensor,
              labels: torch.Tensor, device: torch.device) -> torch.Tensor:
        pass

@dataclass
class PerturbationPipeline:
    """
    Sequence of perturbations to apply in order.
    For example:
      [JpegPerturbation(q=50), FgsmPerturbation(eps=0.03)]
    """
    name: str
    steps: List[Perturbation]

    def apply(self, model: nn.Module, images: torch.Tensor,
              labels: torch.Tensor, device: torch.device) -> torch.Tensor:
        x = images
        for step in self.steps:
            x = step.apply(model, x, labels, device)
        return x

## Registries

In [10]:
# ---------- Compressions registry ----------

@dataclass
class CompressionSpec:
    name: str
    make: Callable[[float], Perturbation]  # param: quality (or similar)

compression_registry: Dict[str, CompressionSpec] = {
    "jpeg": CompressionSpec(
        name="jpeg",
        make=lambda q: JpegPerturbation(quality=int(q)),
    ),
    "jpeg2000": CompressionSpec(
        name="jpeg2000",
        make=lambda q: Jpeg2000Perturbation(quality=float(q)),
    ),
    "pca": CompressionSpec(
        name="pca",
        make=lambda q: PcaPerturbation(quality=float(q)),
    ),
    "patchsvd": CompressionSpec(
        name="patchsvd",
        make=lambda q: PatchSVDPerturbation(quality=float(q), patch_size=8),
    ),
    "lic_roi": CompressionSpec(
        name="lic_roi",
        make=lambda q: LicRoiPerturbation(quality=float(q)),
    ),
}



# ---------- Attacks registry ----------

@dataclass
class AttackSpec:
    name: str
    make: Callable[[float], Perturbation]  # param: epsilon

attack_registry: Dict[str, AttackSpec] = {
    "fgsm": AttackSpec(
        name="fgsm",
        make=lambda eps: FgsmPerturbation(epsilon=eps),
    ),
    "pgd": AttackSpec(
        name="pgd",
        make=lambda eps: PGDPerturbation(epsilon=eps),
    ),
    "apgd": AttackSpec(
        name="apgd",
        make=lambda eps: APGDPerturbation(epsilon=eps, num_steps=20, num_restarts=1, loss_type='dlr'),
    ),
}

## Implementations
#### JPEG

In [11]:
class JpegCompressionPIL:
    def __init__(self, quality: int):
        self.quality = int(quality)

    def __call__(self, img: Image.Image) -> Image.Image:
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=self.quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")


class JpegPerturbation(Perturbation):
    def __init__(self, quality: int):
        super().__init__(name=f"jpeg_q{quality}")
        self.quality = int(quality)
        self.jpeg = JpegCompressionPIL(quality)
        self.to_pil = T.ToPILImage()
        self.to_tensor = T.ToTensor()

    def apply(self, model, images, labels, device):
        images = images.detach().to(device)

        # 1) back to [0,1] pixel space
        pixels = denormalize(images).cpu()

        # 2) apply JPEG in pixel space
        out = []
        for img in pixels:
            pil_img = self.to_pil(img)
            pil_jpeg = self.jpeg(pil_img)
            out.append(self.to_tensor(pil_jpeg))

        out = torch.stack(out).to(device)  # still clamped [0,1]

        # 3) return renormalized
        return renormalize(out)

### JPEG2000

In [12]:
class Jpeg2000CompressionPIL:
    def __init__(self, quality: float, irreversible: bool = True):
        self.quality = float(quality)
        self.irreversible = bool(irreversible)

    @staticmethod
    def quality_to_rate(q: float) -> float:
        """Map quality [1, 100] to JPEG2000 compression ratio.

        In Pillow's quality_mode='rates':
          - 1.0 = lossless
          - >1.0 = lossy (higher = more compression)

        We map:
          q=100 -> rate=1   (lossless)
          q=1   -> rate=100 (heavy compression, ~100x)
        using an exponential scale so mid-range values show clear differences.
        """
        q = float(max(1.0, min(100.0, q)))
        # Invert: low quality = high compression ratio
        lo, hi = 1.0, 100.0
        t = (100.0 - q) / 99.0   # t=0 at q=100 (lossless), t=1 at q=1 (max compression)
        return lo * ((hi / lo) ** t)

    def __call__(self, img: Image.Image) -> Image.Image:
        rate = self.quality_to_rate(self.quality)
        save_kwargs = {
            "quality_mode": "rates",
            "quality_layers": [rate],
            "irreversible": self.irreversible,
        }

        last_exc = None
        for fmt in ("JPEG2000", "JP2"):
            try:
                with BytesIO() as buffer:
                    img.save(buffer, format=fmt, **save_kwargs)
                    buffer.seek(0)
                    out = Image.open(buffer).convert("RGB")
                    out.load()  # decode now so buffer can be released
                    return out
            except Exception as exc:
                last_exc = exc

        raise RuntimeError(
            "JPEG2000 support not available in this Pillow build. "
            "Install OpenJPEG + rebuild Pillow or use an alternative backend."
        ) from last_exc


class Jpeg2000Perturbation(Perturbation):
    def __init__(self, quality: float):
        super().__init__(name=f"jpeg2000_q{int(quality)}")
        self.quality = float(quality)
        self.jpeg2000 = Jpeg2000CompressionPIL(quality=float(quality), irreversible=True)
        self.to_pil = T.ToPILImage()
        self.to_tensor = T.ToTensor()

    def apply(self, model, images, labels, device):
        images = images.detach().to(device)

        # 1) back to [0,1] pixel space
        pixels = denormalize(images).cpu()

        # 2) apply JPEG2000 in pixel space
        out = []
        for img in pixels:
            pil_img = self.to_pil(img)
            pil_jp2 = self.jpeg2000(pil_img)
            out.append(self.to_tensor(pil_jp2))

        out = torch.stack(out).to(device)
        out = torch.clamp(out, 0.0, 1.0)

        # 3) return renormalized
        return renormalize(out)

In [13]:
# Quick sanity check for JPEG2000 perturbation
images_norm, labels = next(iter(base_test_loader))
images_norm = images_norm.to(device)
labels = labels.to(device)

orig_px = denormalize(images_norm)
print(f"orig pixel min/max: {orig_px.min().item():.4f}/{orig_px.max().item():.4f}")

for q in [10, 50, 90]:
    pert = Jpeg2000Perturbation(quality=q)
    pert_norm = pert.apply(model=None, images=images_norm, labels=labels, device=device)
    pert_px = denormalize(pert_norm)

    mse, mae, psnr = batch_metrics_from_normalized(images_norm, pert_norm)

    print(f"[q={q}] shape unchanged: {tuple(pert_norm.shape) == tuple(images_norm.shape)}")
    print(f"[q={q}] dtype unchanged: {pert_norm.dtype == images_norm.dtype}")
    print(f"[q={q}] pert pixel min/max: {pert_px.min().item():.4f}/{pert_px.max().item():.4f}")
    print(
        f"[q={q}] mse={mse.mean().item():.6f}, "
        f"mae={mae.mean().item():.6f}, psnr={psnr.mean().item():.2f}"
    )


orig pixel min/max: 0.0000/1.0000
[q=10] shape unchanged: True
[q=10] dtype unchanged: True
[q=10] pert pixel min/max: 0.0000/1.0000
[q=10] mse=0.049291, mae=0.172289, psnr=13.72
[q=50] shape unchanged: True
[q=50] dtype unchanged: True
[q=50] pert pixel min/max: 0.0000/1.0000
[q=50] mse=0.025872, mae=0.115388, psnr=16.83
[q=90] shape unchanged: True
[q=90] dtype unchanged: True
[q=90] pert pixel min/max: 0.0000/1.0000
[q=90] mse=0.000028, mae=0.003650, psnr=46.75


#### Principal Component Analysis

In [14]:
class PcaPerturbation(Perturbation):

    def __init__(self, quality: float):
        super().__init__(name=f"pca_q{int(quality)}")
        self.quality = float(quality)

    def apply(self, model, images, labels, device):
        images = images.detach().to(device)

        # 1) back to [0,1] pixel space
        x = denormalize(images)
        B, C, H, W = x.shape

        # max possible rank per channel
        k_max = min(H, W)
        # map quality ∈ [0,100] to rank ∈ [1, k_max]
        k = max(1, int(round(self.quality / 100.0 * k_max)))

        # 2) apply PCA (mean-centered SVD) per channel
        x_flat = x.view(B * C, H, W)
        x_out = torch.empty_like(x_flat)

        for i in range(x_flat.size(0)):
            A = x_flat[i]

            # Center rows (subtract row means for true PCA)
            row_mean = A.mean(dim=1, keepdim=True)
            A_centered = A - row_mean

            U, S, Vh = torch.linalg.svd(A_centered, full_matrices=False)

            # take top-k components
            Uk = U[:, :k]
            Sk = S[:k]
            Vhk = Vh[:k, :]

            Ak = (Uk * Sk) @ Vhk

            # Add mean back
            x_out[i] = Ak + row_mean

        # reshape back to [B, C, H, W]
        x_out = x_out.view(B, C, H, W)

        # clamp back into [0,1] just in case
        x_out = torch.clamp(x_out, 0.0, 1.0)

        # 3) return renormalized
        return renormalize(x_out)

#### PatchSVD

In [15]:
class PatchSVDPerturbation(Perturbation):
    """
    Block-based SVD compression (PatchSVD).
    
    1. Splits the image into non-overlapping patches (e.g., 8x8).
    2. Performs SVD on each patch independently.
    3. Retains the top-k singular values based on 'quality'.
    4. Reconstructs the patches and stitches the image back together.
    
    quality: [0, 100]. 
             100 = keep all singular values (lossless reconstruction).
             50  = keep half the singular values.
    patch_size: Size of the square patch (default 8).
    """

    def __init__(self, quality: float, patch_size: int = 8):
        super().__init__(name=f"patchsvd_q{int(quality)}_p{patch_size}")
        self.quality = float(quality)
        self.patch_size = int(patch_size)

    def apply(self, model, images, labels, device):
        images = images.detach().to(device)

        # 1) Move to pixel space [0,1]
        x = denormalize(images)
        B, C, H, W = x.shape
        p = self.patch_size

        # padding
        pad_h = (p - H % p) % p
        pad_w = (p - W % p) % p
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        
        H_pad, W_pad = x.shape[2], x.shape[3]

        # 2) Unfold into patches
        # Output shape: [B, C, GridH, GridW, p, p]
        patches = x.unfold(2, p, p).unfold(3, p, p)
        
        GridH = patches.shape[2]
        GridW = patches.shape[3]

        # 3) Flatten for batch SVD processing
        # Shape: [N_patches, p, p] where N_patches = B * C * GridH * GridW
        patches_flat = patches.contiguous().view(-1, p, p)

        # 4) Apply SVD
        U, S, Vh = torch.linalg.svd(patches_flat, full_matrices=False)

        # Calculate rank k based on quality
        k_max = p
        k = max(1, int(round(self.quality / 100.0 * k_max)))

        Uk = U[:, :, :k]
        Sk = S[:, :k]
        Vhk = Vh[:, :k, :]

        recon_flat = (Uk * Sk.unsqueeze(1)) @ Vhk

        # 5) Refold (Stitch) back to [B, C, GridH, GridW, p, p]
        recon_folded = recon_flat.view(B, C, GridH, GridW, p, p)

        # Permute to [B, C, GridH, p, GridW, p] to prepare for combining
        recon_permuted = recon_folded.permute(0, 1, 2, 4, 3, 5)
        
        # Combine axes to get [B, C, H_pad, W_pad]
        recon = recon_permuted.contiguous().view(B, C, H_pad, W_pad)

        # 6) Crop padding 
        if pad_h > 0 or pad_w > 0:
            recon = recon[:, :, :H, :W]

        # clamp back into [0,1] just in case
        recon = torch.clamp(recon, 0.0, 1.0)

        # 7) return renormalized
        return renormalize(recon)

#### Learned Image Compression (ROI)

In [16]:
# ---- CompressAI model cache ----
_compressai_cache: Dict[tuple, nn.Module] = {}


def _map_quality_to_lic_level(q: float) -> int:
    """Map quality integer (20, 80, etc.) to CompressAI level 1-6."""
    mapping = {20: 2, 80: 5}
    if q in mapping:
        return mapping[q]
    level = int(round(1 + (q - 1) * 5 / 99))
    return max(1, min(6, level))


def _get_compressai_model(quality_level: int, device: torch.device) -> nn.Module:
    key = (quality_level, str(device))
    if key not in _compressai_cache:
        net = cheng2020_attn(quality=quality_level, pretrained=True).to(device).eval()
        net.update()
        _compressai_cache[key] = net
    return _compressai_cache[key]


def get_saliency_map(x_norm: torch.Tensor, model: nn.Module) -> torch.Tensor:
    """Compute spatial saliency via gradient magnitude w.r.t. predicted class.
    Returns mask (B, 1, H, W) in [0, 1]."""
    x = x_norm.detach().requires_grad_(True)
    logits = model(x)
    preds = logits.argmax(dim=1)
    loss = logits[torch.arange(logits.size(0), device=logits.device), preds].sum()
    grad = torch.autograd.grad(loss, x, only_inputs=True)[0]
    saliency = grad.abs().mean(dim=1, keepdim=True)
    B = saliency.size(0)
    flat = saliency.view(B, -1)
    mins = flat.min(dim=1).values.view(B, 1, 1, 1)
    maxs = flat.max(dim=1).values.view(B, 1, 1, 1)
    saliency = (saliency - mins) / (maxs - mins + 1e-8)
    return saliency.detach()


class LicRoiPerturbation(Perturbation):
    def __init__(self, quality: float, roi_weight: float = 1.0):
        super().__init__(name=f"lic_roi_q{int(quality)}")
        self.quality = quality
        self.roi_weight = roi_weight
        self.base_level = _map_quality_to_lic_level(quality)
        self.hq_level = min(6, self.base_level + 1)
        self.lq_level = max(1, self.base_level - 1)

    def _compress_decompress(self, x_pixel: torch.Tensor, level: int, device: torch.device) -> torch.Tensor:
        net = _get_compressai_model(level, device)
        with torch.no_grad():
            out = net(x_pixel)
        return out["x_hat"].clamp(0.0, 1.0)

    def apply(self, model, images, labels, device):
        images = images.detach().to(device)
        if model is None:
            # Fall back to base-level compression with no ROI weighting
            x_pixel = denormalize(images)
            base_recon = self._compress_decompress(x_pixel, self.base_level, device)
            return renormalize(base_recon)

        # Saliency mask from classifier
        mask = get_saliency_map(images, model)

        # Denormalize to [0,1] for CompressAI
        x_pixel = denormalize(images)

        # Pad to multiple of 64 (CompressAI requirement)
        _, _, H, W = x_pixel.shape
        pad_h = (64 - H % 64) % 64
        pad_w = (64 - W % 64) % 64
        if pad_h > 0 or pad_w > 0:
            x_pixel = F.pad(x_pixel, (0, pad_w, 0, pad_h), mode='reflect')

       # HQ and LQ reconstruction passes
        hq_recon = self._compress_decompress(x_pixel, self.hq_level, device)
        lq_recon = self._compress_decompress(x_pixel, self.lq_level, device)

        # Crop padding back to original spatial size
        if pad_h > 0 or pad_w > 0:
            hq_recon = hq_recon[:, :, :H, :W]
            lq_recon = lq_recon[:, :, :H, :W]

        # Resize saliency mask to match spatial dims (handles CIFAR 32x32 vs ImageNet 224x224)
        if mask.shape[2:] != hq_recon.shape[2:]:
            mask = F.interpolate(mask, size=hq_recon.shape[2:], mode='bilinear', align_corners=False)

        # Blend: salient regions get HQ, background gets LQ
        # roi_weight=1.0 means hard switch; values <1.0 soften the boundary
        w = (mask * self.roi_weight).clamp(0.0, 1.0)
        blended = w * hq_recon + (1.0 - w) * lq_recon

        return renormalize(blended)

#### Fast Gradient Sign Method

In [17]:
class FgsmPerturbation(Perturbation):
    def __init__(self, epsilon: float):
        super().__init__(name=f"fgsm_eps{epsilon}")
        self.epsilon = float(epsilon)

    def apply(self, model, images, labels, device):
        model.eval()
        x_norm = images.to(device)
        y = labels.to(device)

        # Move to pixel space [0,1]
        x0_px = denormalize(x_norm).detach()

        # Optimize in pixel space
        x_adv_px = x0_px.clone().detach().requires_grad_(True)

        logits = model(renormalize(x_adv_px))
        loss = torch.nn.functional.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv_px, only_inputs=True)[0]

        x_adv_px = x_adv_px + self.epsilon * grad.sign()
        x_adv_px = project_linf_pixel(x_adv_px, x0_px, self.epsilon)

        return renormalize(x_adv_px.detach())

#### Projected Gradient Descent

In [18]:
class PGDPerturbation(Perturbation):
    def __init__(self, epsilon: float, steps: int = 10, alpha: float = None, random_start: bool = True):
        self.epsilon = float(epsilon)
        self.steps = int(steps)
        self.alpha = float(alpha) if alpha is not None else 2.0 / 255.0
        self.random_start = bool(random_start)
        super().__init__(name=f"pgd_eps{self.epsilon}_k{self.steps}_rs{int(self.random_start)}")

    def apply(self, model, images, labels, device):
        model.eval()
        x_norm = images.to(device)
        y = labels.to(device)

        x0_px = denormalize(x_norm).detach()

        # Optional random start within the L-inf ball
        if self.random_start:
            noise = torch.empty_like(x0_px).uniform_(-self.epsilon, self.epsilon)
            x_adv_px = project_linf_pixel(x0_px + noise, x0_px, self.epsilon).detach()
        else:
            x_adv_px = x0_px.clone().detach()

        for _ in range(self.steps):
            x_adv_px = x_adv_px.detach().requires_grad_(True)

            logits = model(renormalize(x_adv_px))
            loss = torch.nn.functional.cross_entropy(logits, y)
            grad = torch.autograd.grad(loss, x_adv_px, only_inputs=True)[0]

            with torch.no_grad():
                x_adv_px = x_adv_px + self.alpha * grad.sign()
                x_adv_px = project_linf_pixel(x_adv_px, x0_px, self.epsilon)

        return renormalize(x_adv_px.detach())

#### Auto PGD

In [19]:
def dlr_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Difference of Logits Ratio (DLR) loss (untargeted).
    Returned value is intended to be MAXIMIZED by the attack.
    """
    sorted_logits, indices = torch.sort(logits, dim=1, descending=True)

    z_y = logits[torch.arange(logits.shape[0], device=logits.device), labels]
    z_max1 = sorted_logits[:, 0]
    z_max2 = sorted_logits[:, 1]
    z_max3 = sorted_logits[:, 2]

    # If top-1 is the true label, use top-2 as "other"; else top-1 is "other"
    z_other = torch.where(indices[:, 0] == labels, z_max2, z_max1)

    return -(z_y - z_other) / (z_max1 - z_max3 + 1e-12)


class APGDPerturbation(Perturbation):
    """
    Auto-PGD (APGD) style attack with DLR/CE.
    """
    def __init__(
        self,
        epsilon: float = 8/255,
        num_steps: int = 20,
        num_restarts: int = 1,
        loss_type: str = "dlr",
        step_factor: float = 2.0,
    ):
        super().__init__(name=f"apgd_{loss_type}_eps{epsilon}")
        self.epsilon = float(epsilon)
        self.num_steps = int(num_steps)
        self.num_restarts = int(num_restarts)
        self.loss_type = str(loss_type).lower()
        self.step_factor = float(step_factor)

    def apply(self, model, images, labels, device):
        model.eval()
        x_norm0 = images.detach().to(device)
        y = labels.to(device)

        x0_px = denormalize(x_norm0).detach()

        batch_size = x0_px.size(0)

        # Track best found across restarts
        x_best_px = x0_px.clone()
        loss_best = torch.full((batch_size,), -1e10, device=device)

        # Step size schedule checkpoints
        c1 = max(1, int(0.22 * self.num_steps))
        c2 = max(1, int(0.75 * self.num_steps))
        checkpoints = {c1, c2}

        for _ in range(self.num_restarts):
            noise = torch.empty_like(x0_px).uniform_(-self.epsilon, self.epsilon)
            
            x_px = (x0_px + noise).clamp(0.0, 1.0)
            x_px = project_linf_pixel(x_px, x0_px, self.epsilon)

            eta = self.step_factor * self.epsilon

            for i in range(self.num_steps):
                x_px = x_px.detach().requires_grad_(True)

                logits = model(renormalize(x_px))

                if self.loss_type == "ce":
                    loss_indiv = F.cross_entropy(logits, y, reduction='none')
                else:
                    loss_indiv = dlr_loss(logits, y)

                loss = loss_indiv.sum()
                grad = torch.autograd.grad(loss, x_px, only_inputs=True)[0]

                with torch.no_grad():
                    # Update global best per-image
                    improved = loss_indiv > loss_best
                    loss_best[improved] = loss_indiv[improved]
                    x_best_px[improved] = x_px[improved].detach()

                    # Sign step in pixel space
                    x_px = x_px + eta * grad.sign()

                    x_px = project_linf_pixel(x_px, x0_px, self.epsilon)

                    # Step size decay
                    if (i + 1) in checkpoints:
                        eta *= 0.5

        return renormalize(x_best_px.detach())

#### Pipeline builder

In [20]:
@dataclass
class PipelineSpec:
    name: str
    pipeline: PerturbationPipeline
    compression: Optional[str] = None
    attack: Optional[str] = None
    quality: Optional[float] = None
    epsilon: Optional[float] = None

In [21]:
def build_pipeline_grid(
    compression_keys: List[str],
    attack_keys: List[str],
    qualities: List[float],
    epsilons: List[float],
) -> List[PipelineSpec]:

    specs: List[PipelineSpec] = []

    # ---- clean ----
    clean_pipe = PerturbationPipeline(name="clean", steps=[])
    specs.append(PipelineSpec(
        name="clean",
        pipeline=clean_pipe,
        compression=None,
        attack=None,
        quality=None,
        epsilon=None,
    ))

    # ---- compression-only ----
    for ck in compression_keys:
        comp_spec = compression_registry[ck]
        for q in qualities:
            comp = comp_spec.make(q)
            name = f"{comp_spec.name}    ({int(q)}%)"
            pipe = PerturbationPipeline(name=name, steps=[comp])
            specs.append(PipelineSpec(
                name=name,
                pipeline=pipe,
                compression=comp_spec.name,
                attack=None,
                quality=q,
                epsilon=None,
            ))

    # ---- attack-only ----
    for ak in attack_keys:
        atk_spec = attack_registry[ak]
        for eps in epsilons:
            atk = atk_spec.make(eps)
            name = f"{atk_spec.name}    (eps={eps:.2f})"
            pipe = PerturbationPipeline(name=name, steps=[atk])
            specs.append(PipelineSpec(
                name=name,
                pipeline=pipe,
                compression=None,
                attack=atk_spec.name,
                quality=None,
                epsilon=eps,
            ))

    # ---- compression -> attack ----
    for ck in compression_keys:
        comp_spec = compression_registry[ck]
        for ak in attack_keys:
            atk_spec = attack_registry[ak]
            for q in qualities:
                for eps in epsilons:
                    comp = comp_spec.make(q)
                    atk = atk_spec.make(eps)
                    name = f"{comp_spec.name} -> {atk_spec.name}    ({int(q)}%, eps={eps:.2f})"
                    pipe = PerturbationPipeline(name=name, steps=[comp, atk])
                    specs.append(PipelineSpec(
                        name=name,
                        pipeline=pipe,
                        compression=comp_spec.name,
                        attack=atk_spec.name,
                        quality=q,
                        epsilon=eps,
                    ))

    # ---- attack -> compression ----
    for ak in attack_keys:
        atk_spec = attack_registry[ak]
        for ck in compression_keys:
            comp_spec = compression_registry[ck]
            for eps in epsilons:
                for q in qualities:
                    atk = atk_spec.make(eps)
                    comp = comp_spec.make(q)
                    name = f"{atk_spec.name} -> {comp_spec.name}    (eps={eps:.2f}, {int(q)}%)"
                    pipe = PerturbationPipeline(name=name, steps=[atk, comp])
                    specs.append(PipelineSpec(
                        name=name,
                        pipeline=pipe,
                        compression=comp_spec.name,
                        attack=atk_spec.name,
                        quality=q,
                        epsilon=eps,
                    ))

    return specs

# Runner

In [22]:
@torch.no_grad()
def _predict(model, images, device):
    return model(images.to(device))


def evaluate_pipeline(
    model,
    dataloader,
    pipeline: PerturbationPipeline,
    device,
    max_batches=None,
    track_metrics: bool = False,
):
    model.eval()

    # Accumulators for metrics
    total = 0
    correct = 0
    psnr_sum = 0.0
    mse_sum = 0.0
    mae_sum = 0.0
    metric_count = 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        images = images.to(device)
        labels = labels.to(device)

        # Apply perturbations (attacks need gradients enabled)
        if pipeline is not None and len(pipeline.steps) > 0:
            with torch.enable_grad():
                perturbed = pipeline.apply(model, images, labels, device)
        else:
            perturbed = images

        # Metric tracking using helper (Datasets > (Loaders and normalisations))
        if track_metrics:
            batch_mse, batch_mae, batch_psnr = batch_metrics_from_normalized(images, perturbed)
            
            mse_sum += batch_mse.sum().item()
            mae_sum += batch_mae.sum().item()
            psnr_sum += batch_psnr.sum().item()
            metric_count += batch_mse.numel()

        # Normal inference under no_grad
        with torch.no_grad():
            logits = model(perturbed)
            preds = logits.argmax(dim=1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

    result = {
        "accuracy": correct / total,
        "correct": correct,
        "total": total,
    }

    if track_metrics and metric_count > 0:
        result["psnr_mean"] = psnr_sum / metric_count
        result["mse_mean"] = mse_sum / metric_count
        result["mae_mean"] = mae_sum / metric_count
    else:
        result["psnr_mean"] = None
        result["mse_mean"] = None
        result["mae_mean"] = None

    return result

In [23]:
def run_pipeline_grid(
    model_names: List[str],\
    pipeline_specs: List[PipelineSpec],
    dataloader,
    device,
    max_batches=None,
) -> pd.DataFrame:
    records = []

    for model_name in model_names:
        print(f"\n----- Model: {model_name} -----")
        model = build_model(model_name, device)

        for p in pipeline_specs:
            print(f"  Pipeline: {p.name} - started {datetime.datetime.now().time()}")
            res = evaluate_pipeline(
                model=model,
                dataloader=dataloader,
                pipeline=p.pipeline,
                device=device,
                max_batches=max_batches,
                track_metrics=True,
            )

            # Print metrics during operation
            
            print(f"   - Accuracy: {res['accuracy']*100:.2f}%") 
            print(f"   - PSNR:     {res['psnr_mean']:.2f}") 
            print(f"   - MSE:      {res['mse_mean']:.5f}")
            print(f"   - MAE:      {res['mae_mean']:.5f}\n")
            print(f"   - Total:    {res['total']}\n")
            print(f"   - Correct:    {res['correct']}\n")

            

            records.append({
                "model": model_name,
                "pipeline": p.name,
                "compression": p.compression,
                "attack": p.attack,
                "quality": p.quality,
                "epsilon": p.epsilon,
                "accuracy": res["accuracy"],
                "correct": res["correct"],
                "total": res["total"],
                "psnr_mean": res["psnr_mean"],
                "mse_mean": res["mse_mean"],
                "mae_mean": res["mae_mean"],
            })

    return pd.DataFrame.from_records(records)

## Runner Config

In [24]:
compression_keys = ["jpeg"] # , "pca", "patchsvd", "jpeg2000", "lic_roi"
attack_keys = ["apgd"] # "fgsm", "pgd", 

qualities = [40, 44, 48, 52, 56, 60]
epsilons = [8/255]

if DATASET_NAME == "cifar10":
    model_names = ["resnet18_cifar10", "resnet50_cifar10"] # "resnet34_cifar10"
elif DATASET_NAME == "cifar100":
    model_names = ["resnet18_cifar100", "resnet50_cifar100"] # "resnet34_cifar100"
    #model_names = ["vit_base16_cifar100"]
elif DATASET_NAME == "imagenet":
    model_names = ["resnet50_imagenet"]
    
runner_batches_limit = None

## Pipeline builder

In [25]:
pipeline_specs = build_pipeline_grid(
    compression_keys=compression_keys,
    attack_keys=attack_keys,
    qualities=qualities,
    epsilons=epsilons,
)

print("Total pipelines:", len(pipeline_specs))

results_df = run_pipeline_grid(
    model_names=model_names,
    pipeline_specs=pipeline_specs,
    dataloader=base_test_loader,
    device=device,
    max_batches=runner_batches_limit,
)

results_df.head()

Total pipelines: 20

----- Model: resnet18_cifar100 -----


Downloading: "https://huggingface.co/edadaltocg/resnet18_cifar100/resolve/main/pytorch_model.bin" to /root/.cache/torch/hub/checkpoints/resnet18_cifar100.pth
100%|██████████| 42.9M/42.9M [00:00<00:00, 53.4MB/s]

  Pipeline: clean - started 18:18:41.295565


   - Accuracy: 79.26%
   - PSNR:     100.00
   - MSE:      0.00000
   - MAE:      0.00000

   - Total:    10000

   - Correct:    7926

  Pipeline: jpeg    (40%) - started 18:18:43.687575
   - Accuracy: 46.68%
   - PSNR:     27.47
   - MSE:      0.00205
   - MAE:      0.03245

   - Total:    10000

   - Correct:    4668

  Pipeline: jpeg    (44%) - started 18:18:50.416354
   - Accuracy: 48.55%
   - PSNR:     27.79
   - MSE:      0.00192
   - MAE:      0.03130

   - Total:    10000

   - Correct:    4855

  Pipeline: jpeg    (48%) - started 18:18:57.348762
   - Accuracy: 49.93%
   - PSNR:     28.02
   - MSE:      0.00182
   - MAE:      0.03048

   - Total:    10000

   - Correct:    4993

  Pipeline: jpeg    (52%) - started 18:19:04.054695
   - Accuracy: 50.98%
   - PSNR:     28.26
   - MSE:      0.00173
   - MAE:      0.02962

   - Total:    10000

   - Correct:    5098

  Pipeline: jpeg    (56%) - started 18:19:10.844725
   - Accuracy: 52.54%
   - PSNR:     28.51
   - MSE:      0.0016

Downloading: "https://huggingface.co/edadaltocg/resnet50_cifar100/resolve/main/pytorch_model.bin" to /root/.cache/torch/hub/checkpoints/resnet50_cifar100.pth
100%|██████████| 90.7M/90.7M [00:01<00:00, 67.4MB/s]


  Pipeline: clean - started 18:37:08.841162
   - Accuracy: 80.93%
   - PSNR:     100.00
   - MSE:      0.00000
   - MAE:      0.00000

   - Total:    10000

   - Correct:    8093

  Pipeline: jpeg    (40%) - started 18:37:14.225423
   - Accuracy: 45.33%
   - PSNR:     27.47
   - MSE:      0.00205
   - MAE:      0.03245

   - Total:    10000

   - Correct:    4533

  Pipeline: jpeg    (44%) - started 18:37:24.276283
   - Accuracy: 47.64%
   - PSNR:     27.79
   - MSE:      0.00192
   - MAE:      0.03130

   - Total:    10000

   - Correct:    4764

  Pipeline: jpeg    (48%) - started 18:37:34.283973
   - Accuracy: 50.01%
   - PSNR:     28.02
   - MSE:      0.00182
   - MAE:      0.03048

   - Total:    10000

   - Correct:    5001

  Pipeline: jpeg    (52%) - started 18:37:44.459759
   - Accuracy: 51.22%
   - PSNR:     28.26
   - MSE:      0.00173
   - MAE:      0.02962

   - Total:    10000

   - Correct:    5122

  Pipeline: jpeg    (56%) - started 18:37:54.868672
   - Accuracy: 52.29

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,model,pipeline,compression,attack,quality,epsilon,accuracy,correct,total,psnr_mean,mse_mean,mae_mean
0,resnet18_cifar100,clean,None,None,NaN,NaN,0.7926,7926,10000,100.000000,0.000000,0.000000
1,resnet18_cifar100,jpeg (40%),jpeg,None,40.0,NaN,0.4668,4668,10000,27.474945,0.002050,0.032448
2,resnet18_cifar100,jpeg (44%),jpeg,None,44.0,NaN,0.4855,4855,10000,27.787151,0.001916,0.031303
3,resnet18_cifar100,jpeg (48%),jpeg,None,48.0,NaN,0.4993,4993,10000,28.015625,0.001821,0.030482
4,resnet18_cifar100,jpeg (52%),jpeg,None,52.0,NaN,0.5098,5098,10000,28.261434,0.001727,0.029618


In [26]:
results_df.to_excel("results.xlsx", sheet_name="Sheet1")